# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [23]:
from google.colab import userdata
import duckdb

token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{token}'
)
""")

In [6]:
!pip install duckdb

In [7]:
from google.colab import userdata
import duckdb

token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{token}'
)
""")

In [8]:
rel = "hf://datasets/FlyRank/internship-warehouse"

In [9]:
con.sql(f"""
SELECT *
FROM read_parquet(
'{rel}/dim_clients.parquet'
)
LIMIT 5
""")

┌─────────────────────────┬───────────┬────────────────┬────────────────┬───────────────────────────────┬─────────────────────┬─────────────────────┬────────────────┬────────────────┐
│     client_hash_id      │ is_active │ has_gsc_access │ has_ga4_access │        access_profile         │ client_created_date │ client_updated_date │ gsc_data_start │ ga4_data_start │
│         varchar         │  boolean  │    boolean     │    boolean     │            varchar            │        date         │        date         │      date      │      date      │
├─────────────────────────┼───────────┼────────────────┼────────────────┼───────────────────────────────┼─────────────────────┼─────────────────────┼────────────────┼────────────────┤
│ client_04660893ae39614a │ true      │ true           │ true           │ gsc_and_ga4                   │ 2026-04-15          │ 2026-06-27          │ NULL           │ 2026-05-22     │
│ client_05475c07ed21a83a │ true      │ false          │ false          │ no_sea

In [10]:
con.sql(f"""
SELECT COUNT(*)
FROM read_parquet(
'{rel}/dim_clients.parquet'
)
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          104 │
└──────────────┘

In [11]:
con.sql(f"""
SELECT *
FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
LIMIT 5
""")

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_cl

In [16]:
import duckdb
from google.colab import userdata

token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{token}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis
One row represents the daily performance of one content item for one client on one reporting date.

Time window
I use the March 2026 partition (month='2026-03') for development because it is a mid-panel month. I avoid using the final month (June 2026 sample) to prevent developing features inside the natural outcome window.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
WITH march AS (
    SELECT *
    FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03'
)

SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM march;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬────────────┐
│ total_rows │ first_date │ last_date  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘

In [18]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS cnt
FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │  cnt  │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features
gsc_impressions
gsc_clicks
gsc_avg_position
ga4_pageviews
sessions_organic Label: Not defined in this notebook. This notebook focuses on defining and verifying the data contract before modeling.  Context
client_hash_id
content_hash_id
report_date
month

Excluded
Any feature derived from future observations is excluded because it would introduce target leakage and would not be available at prediction time.


In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    sessions_organic
FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
WHERE month='2026-03'
LIMIT 10;
""")

┌─────────────┬─────────────────────────┬──────────────────────────┬─────────────────┬────────────┬───────────────────┬───────────────┬──────────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ gsc_impressions │ gsc_clicks │ gsc_avg_position  │ ga4_pageviews │ sessions_organic │
│    date     │         varchar         │         varchar          │      int64      │   int64    │      double       │     int64     │      int64       │
├─────────────┼─────────────────────────┼──────────────────────────┼─────────────────┼────────────┼───────────────────┼───────────────┼──────────────────┤
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │              20 │          0 │              3.35 │          NULL │             NULL │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_05597932fe4da067 │               1 │          0 │               0.0 │          NULL │             NULL │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_7a105f548d9c6916 │  

In [20]:
con.sql(f"""
SELECT
    AVG(CASE WHEN gsc_impressions IS NULL THEN 1.0 ELSE 0 END) AS miss_impressions,
    AVG(CASE WHEN gsc_clicks IS NULL THEN 1.0 ELSE 0 END) AS miss_clicks,
    AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END) AS miss_position,
    AVG(CASE WHEN ga4_pageviews IS NULL THEN 1.0 ELSE 0 END) AS miss_pageviews,
    AVG(CASE WHEN sessions_organic IS NULL THEN 1.0 ELSE 0 END) AS miss_sessions
FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
WHERE month='2026-03';
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┬─────────────┬────────────────────┬─────────────────────┬─────────────────────┐
│ miss_impressions │ miss_clicks │   miss_position    │   miss_pageviews    │    miss_sessions    │
│      double      │   double    │       double       │       double        │       double        │
├──────────────────┼─────────────┼────────────────────┼─────────────────────┼─────────────────────┤
│              0.0 │         0.0 │ 0.6330736407035681 │ 0.30673966592889734 │ 0.30673966592889734 │
└──────────────────┴─────────────┴────────────────────┴─────────────────────┴─────────────────────┘

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I verify the contract by checking the grain, the row count and date window for the selected month, and the availability of GSC and GA4 data using the provided availability flags.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Grain check: one row should represent one
# report_date × client × content combination.

print("=== Grain Check ===")
print(
    con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS cnt
    FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
    WHERE month='2026-03'
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5;
    """)
)

# 2. Row count and time window

print("\n=== Row Count & Date Window ===")
print(
    con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
    WHERE month='2026-03';
    """)
)

# 3. Availability check

print("\n=== Data Availability ===")
print(
    con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
    WHERE month='2026-03';
    """)
)

=== Grain Check ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │  cnt  │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘


=== Row Count & Date Window ===
┌────────────┬────────────┬────────────┐
│ total_rows │ first_date │ last_date  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘


=== Data Availability ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │
│   int64    │       int64        │       int64        │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │            3611061 │             413966 │
└────────────┴────────────────────┴────────────────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data limitations

Client history is unbalanced because different clients started collecting GSC and GA4 data at different dates.
Rows before GA4 collection begins contain zero-filled GA4 metrics, so ga4_data_available must be checked before analysis.
This notebook uses only a single development month (March 2026), so it does not capture seasonality or long-term trends.
The dataset is pseudonymized, meaning results support analysis but cannot identify real clients or content.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=== Client History Availability ===")
print(
    con.sql(f"""
    SELECT
        access_profile,
        COUNT(*) AS clients,
        MIN(gsc_data_start) AS earliest_gsc,
        MAX(gsc_data_start) AS latest_gsc,
        MIN(ga4_data_start) AS earliest_ga4,
        MAX(ga4_data_start) AS latest_ga4
    FROM read_parquet('{REL}/dim_clients.parquet')
    GROUP BY access_profile
    ORDER BY clients DESC;
    """)
)

=== Client History Availability ===
┌──────────────────────────────────────┬─────────┬──────────────┬────────────┬──────────────┬────────────┐
│            access_profile            │ clients │ earliest_gsc │ latest_gsc │ earliest_ga4 │ latest_ga4 │
│               varchar                │  int64  │     date     │    date    │     date     │    date    │
├──────────────────────────────────────┼─────────┼──────────────┼────────────┼──────────────┼────────────┤
│ gsc_and_ga4                          │      53 │ 2025-01-27   │ 2026-06-02 │ 2025-10-29   │ 2026-06-01 │
│ no_search_or_analytics_access        │      26 │ 2025-11-05   │ 2025-11-05 │ NULL         │ NULL       │
│ gsc_only                             │      14 │ 2025-06-07   │ 2026-04-07 │ NULL         │ NULL       │
│ source_only_missing_client_dimension │      10 │ 2025-11-05   │ 2025-11-07 │ 2026-02-19   │ 2026-02-19 │
│ ga4_only                             │       1 │ NULL         │ NULL       │ NULL         │ NULL       │
└

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.